# Wireless Coverage — Part 4: Naive Tower Viewsheds → H3 Coverage

With the LiDAR-derived surface tables H3-indexed in Part 3, this notebook answers the
core question: *what can a tower actually see?* Place one candidate tower at the centroid
of each H3 cell (the "naive" lattice), compute the line-of-sight viewshed for each tower
over the DSM surface (buildings and tree canopy block the view), and aggregate back to an
H3 per-cell tower-LOS count — a coverage signal that captures how many towers have a clear
sightline to each location.

> **Runtime.** Runs on **Databricks Serverless environment 6** — the lightweight tier (pure
> Python/PySpark, no JAR or GDAL init script). Set `DEMO=True` (default) for a Golden Gate
> Park sub-AOI (~25 towers) to iterate quickly; `DEMO=False` for full San Francisco (~1,150
> towers). Full SF is the shipped deliverable.

---
**Last Update:** 2026-10-03

In [ ]:
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall \
  "geobrix[light_env6,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
%pip install --quiet \
  "geobrix[light_env6,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"

In [ ]:
%restart_python

In [ ]:
import math, time, inspect, warnings
warnings.filterwarnings("ignore")
from pyspark.sql import functions as F
from pyspark.databricks.sql import functions as DBF

from databricks.labs.gbx.pyrx import functions as rx
from databricks.labs.gbx.pygx import functions as gx
from databricks.labs.gbx.ds.register import register
from databricks.labs.gbx.vizx import (
    plot_static,
    cells_as_gdf, vector_layer,
)

rx.register(spark)
gx.register(spark)
register(spark)    # installs gbx_rst_* SQL UDFs including gbx_rst_h3_rastertogridmax

## T0 — Preconditions: API verification and dependency check

This section runs live checks on env6 before any implementation cells. Findings are
recorded in `.superpowers/sdd/2026-10-02-wireless-coverage-nb4/t0-preconditions.md`.

In [ ]:
# T0.1 — Confirm input tables exist
for _tbl in [
    "geospatial_docs.geobrix.wc_surface_dsm",
    "geospatial_docs.geobrix.wc_surface_dtm",
    "geospatial_docs.geobrix.wc_h3_dem",
    "geospatial_docs.geobrix.wc_h3_dsm",
]:
    assert spark.catalog.tableExists(_tbl), f"BLOCKER: table missing — run nb2/nb3 first: {_tbl}"
print("T0.1 — All required input tables exist: OK")

In [ ]:
# T0.2 — h3_centeraswkb on Serverless env6 (Photon/Databricks SQL check)
# Returns: _CENTER_FN = 'product' if h3_centeraswkb works; 'h3_python' as fallback.
try:
    spark.range(1).select(DBF.h3_centeraswkb(F.lit(599686042433355775))).limit(1).collect()
    print("T0.2 — h3_centeraswkb: OK (Photon/Databricks SQL available on this env)")
    _CENTER_FN = "product"     # use DBF.h3_centeraswkb
except Exception as _e:
    print(f"T0.2 — h3_centeraswkb NOT available ({type(_e).__name__}: {_e}); using h3 Python fallback")
    _CENTER_FN = "h3_python"   # use h3.cell_to_latlng UDF
print(f"  _CENTER_FN = '{_CENTER_FN}'")

In [ ]:
# T0.3 — rst_viewshed signature: confirm 'crs' parameter exists
sig_vs = inspect.signature(rx.rst_viewshed)
assert "crs" in sig_vs.parameters, "BLOCKER: rst_viewshed has no 'crs' parameter"
assert "materialize" in sig_vs.parameters, "BLOCKER: rst_viewshed has no 'materialize' parameter"
print(f"T0.3 — rst_viewshed signature: {sig_vs}")
print("  crs + materialize parameters confirmed: OK")

In [ ]:
# T0.4 — rst_transform: check for resolution/scale override (GAP check — expect ABSENT)
sig_tx = inspect.signature(rx.rst_transform)
print(f"T0.4 — rst_transform signature: {sig_tx}")
_has_res_override = any(
    p in sig_tx.parameters
    for p in ("resolution", "scale", "pixel_size", "scale_m", "size")
)
if _has_res_override:
    print("  rst_transform HAS a resolution override — Ruling R12 can use it.")
else:
    print(
        "  GAP CONFIRMED: rst_transform has NO resolution override. "
        "Ruling R12 → Option A: accept native-res viewshed; profile LATERAL in T6."
    )

In [ ]:
# T0.5 — rst_merge_agg and rst_sample callable
assert callable(rx.rst_merge_agg), "BLOCKER: rst_merge_agg not callable"
assert callable(rx.rst_sample),    "BLOCKER: rst_sample not callable"
print("T0.5 — rst_merge_agg: OK | rst_sample: OK")

In [ ]:
# T0.6 — gbx_rst_h3_rastertogridmax LATERAL on one DSM tile (functional check)
# ANSI LATERAL form: FROM <table>, LATERAL <udtf>(...) <alias>
# (comma after table is required; matches nb3 verified pattern)
_dsm1 = (
    spark.table("geospatial_docs.geobrix.wc_surface_dsm")
    .limit(1)
    .select(rx.rst_transform("dsm", F.lit(4326)).alias("dsm_4326"))
)
_dsm1.createOrReplaceTempView("_t0_dsm")
_r = spark.sql("""
    SELECT t.cellID, t.measure, t.band
    FROM   _t0_dsm,
           LATERAL gbx_rst_h3_rastertogridmax(dsm_4326, 9) t
    LIMIT 5
""").collect()
assert len(_r) > 0, "BLOCKER: gbx_rst_h3_rastertogridmax returns no rows on DSM tile"
print(f"T0.6 — gbx_rst_h3_rastertogridmax LATERAL on DSM: OK ({len(_r)} sample rows)")
print(f"  Sample: cellID={_r[0]['cellID']}, measure={_r[0]['measure']:.2f}, band={_r[0]['band']}")

In [ ]:
# T0.7 — wc_surface_dsm schema and AOI origin
print("T0.7 — wc_surface_dsm schema:")
spark.table("geospatial_docs.geobrix.wc_surface_dsm").printSchema()

from databricks.labs.gbx.sample.lidar import aoi_lonlat_to_3857 as _aoi_fn
_SF_AOI_CHECK = (-122.55, 37.70, -122.35, 37.85)
_X0c, _Y0c, _X1c, _Y1c = _aoi_fn(*_SF_AOI_CHECK)
print(f"  AOI 3857 origin: X0={_X0c:.0f} Y0={_Y0c:.0f} X1={_X1c:.0f} Y1={_Y1c:.0f}")
print("T0.7 — Schema and AOI origin: OK")

## T1 — Parameters and source table loads

In [ ]:
# --- catalog / schema (shared with nb3) ---
CATALOG      = "geospatial_docs"
SCHEMA       = "geobrix"
OUT_PREFIX   = "wc_h3"           # wc_h3_coverage

# --- DSM / DTM source surfaces (from nb2) ---
DSM_TABLE    = f"{CATALOG}.{SCHEMA}.wc_surface_dsm"
DTM_TABLE    = f"{CATALOG}.{SCHEMA}.wc_surface_dtm"
# H3-gridded DEM (from nb3) — used for tower-elevation reference
DEM_H3_TABLE = f"{CATALOG}.{SCHEMA}.wc_h3_dem"

# --- viewshed parameters (all configurable) ---
TOWER_RES        = 9          # H3 resolution for tower lattice AND coverage output
TOWER_HEIGHT_M   = 30.0       # tower height above ground (DEM), metres
TARGET_HEIGHT_M  = 1.6        # target height above DSM (person at ground level)
# Ruling R11: MAX_DISTANCE_M = 3000 m default.
# At 1 m native resolution, a 5 km radius → merged tile ~10000×10000 px = ~100 MB; this
# OOMs Serverless workers (64 MB per-task cap in materialize_decision). At 3 km, the merged
# tile is ~6000×6000 px ≈ 36 MB uncompressed; safely within cap. Keep 5000 m as an
# override for coarser-resolution DSM or classic clusters with higher memory budgets.
MAX_DISTANCE_M   = 3000.0     # viewshed radius cap, metres (Ruling R11)

# --- SF AOI (lon/lat BBOX) — must match nb2/nb3 AOI exactly ---
SF_AOI  = (-122.55, 37.70, -122.35, 37.85)
TILE_M  = 1024.0              # DSM tile size in metres (must match nb2)

# --- runtime scope ---
DEMO         = True    # True → Golden Gate Park ~25 towers; False → full SF ~1,150 towers
FORCE_RELOAD = False   # True → overwrite wc_h3_coverage; False → reuse if exists
INTERACTIVE  = False   # True → deck.gl live render; False → static (GitHub-safe)

# --- fan-out tuning ---
N_PARTITIONS = 50 if DEMO else 200   # repartition(N, 'tower_cellid') for viewshed fan-out

In [ ]:
from databricks.labs.gbx.sample.lidar import aoi_lonlat_to_3857, lonlat_to_3857

X0, Y0, X1, Y1 = aoi_lonlat_to_3857(*SF_AOI)
print(f"AOI EPSG:3857 origin: X0={X0:.0f} Y0={Y0:.0f}")
print(f"AOI extent: {X1 - X0:.0f} m × {Y1 - Y0:.0f} m")

if DEMO:
    # Golden Gate Park centroid tile window (±2 tiles, same as nb3 DEMO_FILTER)
    _gx, _gy = lonlat_to_3857(-122.486, 37.769)
    DEMO_TX = int((_gx - X0) / TILE_M)
    DEMO_TY = int((_gy - Y0) / TILE_M)
    DEMO_RADIUS_TILES = 2
    print(
        f"DEMO tile window: tx∈[{DEMO_TX - DEMO_RADIUS_TILES},{DEMO_TX + DEMO_RADIUS_TILES}] "
        f"ty∈[{DEMO_TY - DEMO_RADIUS_TILES},{DEMO_TY + DEMO_RADIUS_TILES}]"
    )

In [ ]:
def _persist(df, name):
    """Write to ``CATALOG.SCHEMA.OUT_PREFIX_<name>`` and return it read back.
    FORCE_RELOAD rebuilds; otherwise an existing table is reused."""
    tbl = f"{CATALOG}.{SCHEMA}.{OUT_PREFIX}_{name}"
    if FORCE_RELOAD or not spark.catalog.tableExists(tbl):
        df.write.mode("overwrite").saveAsTable(tbl)
    return spark.table(tbl)

In [ ]:
dsm_full = spark.table(DSM_TABLE)
dtm_full = spark.table(DTM_TABLE)
if DEMO:
    DEMO_FILTER = (
        F.col("tx").between(DEMO_TX - DEMO_RADIUS_TILES, DEMO_TX + DEMO_RADIUS_TILES) &
        F.col("ty").between(DEMO_TY - DEMO_RADIUS_TILES, DEMO_TY + DEMO_RADIUS_TILES)
    )
    dsm_in = dsm_full.where(DEMO_FILTER)
    dtm_in = dtm_full.where(DEMO_FILTER)
else:
    dsm_in, dtm_in = dsm_full, dtm_full
n_dsm = dsm_in.count()
n_dtm = dtm_in.count()
print(f"DSM tiles: {n_dsm} | DTM tiles: {n_dtm}")
assert n_dsm > 0, "No DSM tiles loaded — check DEMO filter or nb2 tables"
assert n_dtm > 0, "No DTM tiles loaded"

In [ ]:
from shapely.geometry import box as _box
from shapely import wkb as _wkb
import pyproj

W, S, E, N = SF_AOI
if DEMO:
    # DEMO AOI: tile window extent in lon/lat
    _t4326 = pyproj.Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
    _tx_min_m = X0 + (DEMO_TX - DEMO_RADIUS_TILES) * TILE_M
    _tx_max_m = X0 + (DEMO_TX + DEMO_RADIUS_TILES + 1) * TILE_M
    _ty_min_m = Y0 + (DEMO_TY - DEMO_RADIUS_TILES) * TILE_M
    _ty_max_m = Y0 + (DEMO_TY + DEMO_RADIUS_TILES + 1) * TILE_M
    W, S = _t4326.transform(_tx_min_m, _ty_min_m)
    E, N = _t4326.transform(_tx_max_m, _ty_max_m)
_aoi_geom = _box(W, S, E, N)
AOI_WKB = _aoi_geom.wkb
print(f"Tower lattice AOI: ({W:.4f}, {S:.4f}, {E:.4f}, {N:.4f})")

## T2 — Tower lattice: H3 cells at TOWER_RES covering the AOI

In [ ]:
# h3_try_coverash3(geom_wkb, res) returns ARRAY<LONG> of H3 cell IDs (non-exception variant).
# Confirmed: h3_rasterize_isobands.ipynb lines 759, 811.
tower_cells = (
    spark.createDataFrame([(1,)], "g int")
    .select(
        F.explode(
            DBF.h3_try_coverash3(F.lit(AOI_WKB), F.lit(TOWER_RES))
        ).alias("tower_cellid")
    )
    .where(F.col("tower_cellid").isNotNull())
)
n_towers = tower_cells.count()
print(f"Tower lattice: {n_towers} H3 cells at res {TOWER_RES} covering the AOI")
# Expected: ~25 (DEMO) / ~1,150 (full SF) at res 9

In [ ]:
# H3 centroid WKB and lon/lat extraction.
# h3_centeraswkb(cellid) → WKB POINT in EPSG:4326.
# If Photon unavailable (T0 flagged _CENTER_FN == 'h3_python'), use the h3 UDF path.

if _CENTER_FN == "product":
    towers_wkb = tower_cells.withColumn(
        "center_wkb", DBF.h3_centeraswkb("tower_cellid")
    )
    # Extract lon/lat from the WKB point for subsequent 3857 arithmetic.
    # DBF.st_geomfromwkb + st_x/st_y confirmed in vapor-eyes (cells 62-64).
    towers_latlon = (
        towers_wkb
        .withColumn("_geom", DBF.st_geomfromwkb("center_wkb"))
        .withColumn("lon",   DBF.st_x("_geom"))
        .withColumn("lat",   DBF.st_y("_geom"))
        .drop("_geom")
    )
else:
    # h3 Python UDF fallback: h3.cell_to_latlng returns (lat, lng)
    import h3 as _h3lib
    from pyspark.sql.types import StructType, StructField, DoubleType, BinaryType

    @F.udf(StructType([
        StructField("lat", DoubleType()),
        StructField("lon", DoubleType()),
        StructField("wkb", BinaryType()),
    ]))
    def _h3_center(cellid):
        from shapely.geometry import Point
        from shapely import wkb as _wkb2
        lat, lng = _h3lib.cell_to_latlng(cellid)
        return {"lat": lat, "lon": lng, "wkb": _wkb2.dumps(Point(lng, lat))}

    towers_latlon = (
        tower_cells
        .withColumn("_c", _h3_center("tower_cellid"))
        .withColumn("lat", F.col("_c.lat"))
        .withColumn("lon", F.col("_c.lon"))
        .withColumn("center_wkb", F.col("_c.wkb"))
        .drop("_c")
    )

towers_latlon.show(5)

In [ ]:
# lon/lat → EPSG:3857 UDF (Ruling R4 — pure-math Mercator, avoids st_transform Photon dependency).
# Reuses the same Mercator formula as databricks.labs.gbx.sample.lidar.lonlat_to_3857
# (verified: _MERC_R = 20037508.342789244; same constants) for cross-series consistency.
_MERC_R = 20037508.342789244

from pyspark.sql.types import StructType, StructField, DoubleType as _DT

@F.udf(StructType([StructField("x", _DT()), StructField("y", _DT())]))
def _lonlat_to_3857(lon, lat):
    """Spherical Mercator (EPSG:3857): same formula as sample.lidar.lonlat_to_3857."""
    import math as _math
    x = lon * _MERC_R / 180.0
    lat_r = _math.radians(lat)
    y = _math.log(_math.tan(lat_r / 2.0 + _math.pi / 4.0)) * (_MERC_R / _math.pi)
    return {"x": x, "y": y}

towers_df = (
    towers_latlon
    .withColumn("_xy",     _lonlat_to_3857("lon", "lat"))
    .withColumn("tower_x", F.col("_xy.x"))   # EPSG:3857 x, metres
    .withColumn("tower_y", F.col("_xy.y"))   # EPSG:3857 y, metres
    .drop("_xy")
)
towers_df.show(5)

In [ ]:
# Sanity: all tower_x/y in SF's expected EPSG:3857 range
_tx_stats = towers_df.selectExpr(
    "min(tower_x)", "max(tower_x)", "min(tower_y)", "max(tower_y)"
).first()
print(
    f"Tower 3857 range: x=[{_tx_stats[0]:.0f}, {_tx_stats[1]:.0f}] "
    f"y=[{_tx_stats[2]:.0f}, {_tx_stats[3]:.0f}]"
)
# SF 3857: x ≈ -13647000 to -13627000, y ≈ 4539000 to 4557000
assert _tx_stats[0] < -13620000, f"tower_x too large ({_tx_stats[0]:.0f}) — CRS conversion likely wrong"
assert _tx_stats[3] < 4560000,   f"tower_y max too large ({_tx_stats[3]:.0f})"
assert _tx_stats[2] > 4535000,   f"tower_y min too small ({_tx_stats[2]:.0f})"
assert towers_df.count() == n_towers, "Tower count changed after 3857 conversion"
print(f"Tower 3857 coordinates — sanity check OK (n={n_towers})")

## T3 — Cross-tile DSM windowing helper (TDD)

For each tower, range-join to DSM tiles whose tx/ty index falls within `MAX_DISTANCE_M`, then
merge into one DSM tile per tower via `rst_merge_agg`. Arithmetic tile-index range join avoids
any ST_ geometry dependency (Ruling R3).

In [ ]:
# Unit test: tile-range arithmetic (pure Python, no Spark required — TDD gate).
def _tile_range(coord_3857, origin_3857, radius, tile_size):
    """Integer tx range for a 1D window. Returns (tx_min, tx_max) inclusive."""
    tx_min = math.floor((coord_3857 - origin_3857 - radius) / tile_size)
    tx_max = math.floor((coord_3857 - origin_3857 + radius) / tile_size)
    return int(tx_min), int(tx_max)

# Test 1: tower at x=1000, X0=0, TILE_M=100, R=150
# → tx_min = floor((1000-0-150)/100) = floor(8.5) = 8
# → tx_max = floor((1000-0+150)/100) = floor(11.5) = 11
assert _tile_range(1000, 0, 150, 100) == (8, 11), f"Test 1 failed: got {_tile_range(1000, 0, 150, 100)}"

# Test 2: tower exactly on tile boundary (x=1024, TILE_M=1024, R=5000, X0=0)
# → tx_min = floor((1024-0-5000)/1024) = floor(-3.88) = -4
# → tx_max = floor((1024-0+5000)/1024) = floor(5.88) = 5
assert _tile_range(1024, 0, 5000, 1024) == (-4, 5), f"Test 2 failed: got {_tile_range(1024, 0, 5000, 1024)}"

# Test 3: centred in a tile (x=512, X0=0, TILE_M=1024, R=1024)
# → tx_min = floor((512-0-1024)/1024) = floor(-0.5) = -1
# → tx_max = floor((512-0+1024)/1024) = floor(1.5) = 1
assert _tile_range(512, 0, 1024, 1024) == (-1, 1), f"Test 3 failed: got {_tile_range(512, 0, 1024, 1024)}"

print("_tile_range unit tests PASSED (3/3)")

In [ ]:
def _window_tiles(tower_df, dsm_df, max_dist_m, x0, y0, tile_m):
    """Range-join towers to DSM tiles and merge into one tile per tower.

    Args:
        tower_df:   DataFrame with (tower_cellid, tower_x, tower_y, center_wkb) in EPSG:3857.
        dsm_df:     DataFrame with (tx INT, ty INT, dsm STRUCT) — EPSG:3857 tiles.
        max_dist_m: Viewshed radius (metres).
        x0, y0:     EPSG:3857 AOI origin (X0, Y0 from aoi_lonlat_to_3857).
        tile_m:     Tile size in metres (TILE_M parameter).

    Returns:
        DataFrame with (tower_cellid, center_wkb, tower_x, tower_y, dsm_window) —
        one merged tile per tower. ``dsm_window`` spans the merged extent of all
        intersecting tiles.

    Notes:
        - Uses rst_merge_agg (group aggregator) — safe for FILE tiles on Spark 4.0
          (avoids collect_list + rst_merge nondeterminism issue, pyrx:8862).
        - Tile at (tx, ty) intersects the tower window iff:
            tx >= floor((tower_x - x0 - R) / tile_m)  AND
            tx <= floor((tower_x - x0 + R) / tile_m)
          (same for ty). This is the Ruling R3 arithmetic range join.
    """
    R = max_dist_m
    towers_ranged = tower_df.select(
        "tower_cellid",
        "tower_x", "tower_y",
        "center_wkb",
        F.floor((F.col("tower_x") - x0 - R) / tile_m).cast("int").alias("tx_min"),
        F.floor((F.col("tower_x") - x0 + R) / tile_m).cast("int").alias("tx_max"),
        F.floor((F.col("tower_y") - y0 - R) / tile_m).cast("int").alias("ty_min"),
        F.floor((F.col("tower_y") - y0 + R) / tile_m).cast("int").alias("ty_max"),
    )
    # Range join: include DSM tiles whose (tx, ty) falls within the tower's window.
    joined = towers_ranged.join(
        dsm_df,
        (
            (F.col("tx") >= F.col("tx_min")) &
            (F.col("tx") <= F.col("tx_max")) &
            (F.col("ty") >= F.col("ty_min")) &
            (F.col("ty") <= F.col("ty_max"))
        ),
        "inner",
    )
    # One merged DSM tile per tower — rst_merge_agg is safe for FILE tiles on Spark 4.
    # (Ruling R2: group aggregator, not collect_list + rst_merge.)
    windowed = joined.groupBy("tower_cellid", "center_wkb", "tower_x", "tower_y").agg(
        rx.rst_merge_agg("dsm").alias("dsm_window")
    )
    return windowed

In [ ]:
# Windowing cross-tile verification: check that at least one tower produces a
# multi-tile window (not a single-tile no-op), proving the range-join works.
_R = MAX_DISTANCE_M
_towers_ranged_check = towers_df.select(
    "tower_cellid",
    F.floor((F.col("tower_x") - X0 - _R) / TILE_M).cast("int").alias("tx_min"),
    F.floor((F.col("tower_x") - X0 + _R) / TILE_M).cast("int").alias("tx_max"),
    F.floor((F.col("tower_y") - Y0 - _R) / TILE_M).cast("int").alias("ty_min"),
    F.floor((F.col("tower_y") - Y0 + _R) / TILE_M).cast("int").alias("ty_max"),
).withColumn(
    "tiles_in_window",
    (F.col("tx_max") - F.col("tx_min") + 1) * (F.col("ty_max") - F.col("ty_min") + 1)
)
_win_stats = _towers_ranged_check.selectExpr(
    "min(tiles_in_window) as min_tiles",
    "max(tiles_in_window) as max_tiles",
    "avg(tiles_in_window) as avg_tiles",
).first()
print(
    f"Window sizes: min={_win_stats['min_tiles']} max={_win_stats['max_tiles']} "
    f"avg={_win_stats['avg_tiles']:.1f} tiles/tower"
)
# At 5 km radius / 1024 m tile, each tower should span (10000/1024+1)^2 ≈ ~100 tiles.
# (In DEMO the DSM set is only 25 tiles, so actual joined tiles = min(window, available).)
assert _win_stats['max_tiles'] > 1, (
    "All towers map to a single tile — range-join is not spanning multiple tiles. "
    "Check TILE_M and MAX_DISTANCE_M values."
)
print("Cross-tile windowing geometry check: PASSED — windows span multiple tiles")

In [ ]:
_t0_win = time.time()
dsm_windows_df = _window_tiles(towers_df, dsm_in, MAX_DISTANCE_M, X0, Y0, TILE_M)
dsm_windows_df = dsm_windows_df.repartition(N_PARTITIONS, "tower_cellid")
# Ruling R13: do NOT .cache() merged raster tiles on Serverless.
# .cache() triggers immediate materialization of all merged tiles simultaneously;
# at 3 km radius / 1 m px, each tile is ~36 MB; caching 25+ tiles OOMs workers.
# dsm_windows_df is lazy — the merge runs per-tower on demand in T4 and T5.
# Count windows using the pre-merge range join (avoids triggering rst_merge_agg for count).
_R = MAX_DISTANCE_M
_n_windows_check = (
    towers_df.select(
        "tower_cellid",
        F.floor((F.col("tower_x") - X0 - _R) / TILE_M).cast("int").alias("tx_min"),
        F.floor((F.col("tower_x") - X0 + _R) / TILE_M).cast("int").alias("tx_max"),
        F.floor((F.col("tower_y") - Y0 - _R) / TILE_M).cast("int").alias("ty_min"),
        F.floor((F.col("tower_y") - Y0 + _R) / TILE_M).cast("int").alias("ty_max"),
    )
    .join(dsm_in, (
        (F.col("tx") >= F.col("tx_min")) & (F.col("tx") <= F.col("tx_max")) &
        (F.col("ty") >= F.col("ty_min")) & (F.col("ty") <= F.col("ty_max"))
    ), "inner")
    .select("tower_cellid").distinct()
)
n_windows = _n_windows_check.count()
print(f"DSM windows: {n_windows} towers have DSM coverage in {time.time() - _t0_win:.1f}s")
assert n_windows == n_towers, (
    f"Expected {n_towers} towers with DSM coverage, got {n_windows}. "
    "Missing → some tower centroids fall outside the DSM extent."
)

In [ ]:
# Merged tile size note (Ruling R11/R13).
# MAX_DISTANCE_M = 3000 m → estimated merged tile size: ~6000×6000 px = ~36 MB uncompressed.
# At GDAL DEFLATE compression (typical for DSM tiles), expect ~5-15 MB per tile.
# This is within the 64 MB Serverless per-task materialization cap.
# Actual tile size is measured in T5 when rst_viewshed materializes each tile.
# dsm_windows_df is lazy — no merge happens here.
_est_px = int((2 * MAX_DISTANCE_M + TILE_M) ** 2)
_est_mb_uncompressed = _est_px / 1024 / 1024   # uint8 = 1 byte/px
print(
    f"Estimated merged tile footprint: ~{int(2*MAX_DISTANCE_M+TILE_M)}×{int(2*MAX_DISTANCE_M+TILE_M)} px "
    f"= ~{_est_mb_uncompressed:.0f} MB uncompressed. "
    f"Expect ~5-15 MB compressed (DEFLATE)."
)
if _est_mb_uncompressed > 64:
    print(
        f"  WARNING: uncompressed estimate exceeds 64 MB Serverless cap. "
        "Reduce MAX_DISTANCE_M if rst_merge_agg OOMs in T5."
    )
print("dsm_windows_df ready (lazy — merge runs per-tower in T4/T5)")

## T4 — Tower elevation: observer height above the DSM

Each tower is modelled as a point `TOWER_HEIGHT_M` metres above the bare-earth DEM. Because
`rst_viewshed` measures `observer_height` *above the surface raster* (the DSM), the effective
height passed is:

    observer_height = (DEM_elevation + TOWER_HEIGHT_M) − DSM_elevation_at_tower

Where DSM elevation ≈ DEM at open ground sites; the correction matters mainly for towers
that would otherwise sit inside buildings or tree canopies. Values are clamped to 0 (a
tower below the DSM surface is treated as ground level).

**CRS note:** `rst_sample` with `crs=4326` reprojects the observer point (EPSG:4326 WKB
from `h3_centeraswkb`) to the tile CRS (3857) internally — the same mechanism as Ruling R5
for `rst_viewshed`.

In [ ]:
# Join towers to DTM home tile (the tile containing the tower centroid — 1:1 join).
# home_tx / home_ty = the tile whose lower-left corner bounds the tower centroid.
towers_with_tile = towers_df.select(
    "tower_cellid", "center_wkb", "tower_x", "tower_y",
    "lon", "lat",    # carry through for diagnostics
    F.floor((F.col("tower_x") - X0) / TILE_M).cast("int").alias("home_tx"),
    F.floor((F.col("tower_y") - Y0) / TILE_M).cast("int").alias("home_ty"),
)
dtm_keyed = dtm_in.select(
    F.col("tx").alias("home_tx"),
    F.col("ty").alias("home_ty"),
    F.col("dtm").alias("dtm_tile"),
)
towers_dtm = towers_with_tile.join(dtm_keyed, ["home_tx", "home_ty"], "left")

# rst_sample(tile, geom, crs=4326): returns ARRAY<DOUBLE>; .getItem(0) for band 1.
# crs=4326 → observer point (EPSG:4326 WKB from h3_centeraswkb) reprojected to tile CRS (3857).
towers_sampled = towers_dtm.withColumn(
    "dem_elev",
    rx.rst_sample("dtm_tile", "center_wkb", F.lit(4326)).getItem(0),
).drop("dtm_tile")

In [ ]:
# Sample DSM elevation at tower centroid from the HOME tile (single-tile, no merge).
# The merged dsm_windows_df (rst_merge_agg) is reserved for T5 rst_viewshed.
# For a point sample at the centroid, the home tile is sufficient and avoids
# triggering the full merge in T4 (Ruling R13: merge is OOM on Serverless when
# chained into a T4 aggregate action).
dsm_keyed = dsm_in.select(
    F.col("tx").alias("home_tx"),
    F.col("ty").alias("home_ty"),
    F.col("dsm").alias("dsm_home_tile"),
)
towers_sampled = (
    towers_sampled
    .join(dsm_keyed, ["home_tx", "home_ty"], "left")
    .withColumn(
        "dsm_elev",
        rx.rst_sample("dsm_home_tile", "center_wkb", F.lit(4326)).getItem(0),
    )
    .drop("dsm_home_tile")
)

In [ ]:
# observer_height = max(0, DEM_elev + TOWER_HEIGHT_M - DSM_elev)
# Clamp to 0: a tower inside a building/canopy is treated as ground level.
towers_with_height = towers_sampled.withColumn(
    "observer_height",
    F.greatest(
        F.lit(0.0),
        F.col("dem_elev") + F.lit(TOWER_HEIGHT_M) - F.col("dsm_elev"),
    ),
)

# Diagnostics: observer_height distribution
_oh_stats = towers_with_height.selectExpr(
    "count(*) as n_towers",
    "sum(case when observer_height = 0.0 then 1 else 0 end) as n_inside_dsm",
    "min(observer_height) as min_oh",
    "max(observer_height) as max_oh",
    "avg(observer_height) as avg_oh",
).first()
print(
    f"Observer height stats: min={_oh_stats['min_oh']:.1f} m  "
    f"max={_oh_stats['max_oh']:.1f} m  avg={_oh_stats['avg_oh']:.1f} m  "
    f"towers inside DSM surface (oh=0): {_oh_stats['n_inside_dsm']}"
)

In [ ]:
# Handle null elevations: towers where DEM or DSM sampling fails (tile boundary edge
# cases) fall back to TOWER_HEIGHT_M above a flat surface.
towers_with_height = towers_with_height.withColumn(
    "observer_height",
    F.when(
        F.col("dem_elev").isNull() | F.col("dsm_elev").isNull(),
        F.lit(TOWER_HEIGHT_M),   # assume flat ground if sampling fails
    ).otherwise(F.col("observer_height")),
)
n_null_elev = towers_with_height.where(
    F.col("dem_elev").isNull() | F.col("dsm_elev").isNull()
).count()
if n_null_elev > 0:
    print(
        f"WARNING: {n_null_elev} towers have null DEM/DSM elevation — "
        f"using TOWER_HEIGHT_M={TOWER_HEIGHT_M} m fallback."
    )

# Final validation
n_final = towers_with_height.count()
n_null_oh = towers_with_height.where(F.col("observer_height").isNull()).count()
assert n_final == n_towers, f"Expected {n_towers} rows, got {n_final}"
assert n_null_oh == 0, f"{n_null_oh} towers still have null observer_height after fallback"
print(
    f"T4 validation PASSED: {n_final} towers with non-null observer_height | "
    f"{n_null_elev} null-elevation fallbacks"
)
towers_with_height.select(
    "tower_cellid", "lon", "lat", "dem_elev", "dsm_elev", "observer_height"
).show(5)